# 第 7 章实践：SIMT Gather（Embedding 场景）


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 客观题

1. （填空）Gather 的输出行号连续、读取行号由 ______ 决定；Scatter 的风险是 ______。
2. （选择）grid-stride 循环的步长是：
   A. block_dim().x<br/>
   B. grid_dim().x × block_dim().x<br/>
   C. 数据总量
3. （判断）`hist[0,b] = hist[0,b] + 1` 在多线程下是安全的。（对/错）
4. （多选）以下哪些在 SIMT 函数内不支持？
   A. pl.simt.atomic_add<br/>
   B. pl.make_tile_group<br/>
   C. 动态 GM Shape 的 Tensor<br/>
   D. pl.simt.syncthreads

In [ ]:
!cat ./answer/07.05_objective_answer.txt

## 2. 编程实践一（简单）：小规模 Gather

**任务**：实现 Embedding Gather：input [1024, 64] FP32、indices [1, 512] INT32（值域 [0,1024)）、output [512, 64]。与 `input.index_select(0, indices.reshape(-1).to(torch.int64))` 的 PyTorch 结果对照（rtol=0, atol=0）。


In [ ]:
# 在此完成实践一
pass

## 3. 编程实践二（中等）：带 top-k 的 Gather（大模型检索场景雏形）

**任务**：每行数据有一个分数（scores [1, 1024] FP32），从 1024 行中选出分数最高的 256 行（用 torch.topk 预计算索引，SIMT Kernel 只负责 Gather），Gather 到 output [256, 64]。要求：grid-stride 版本、8 核启动。

**思考**：参考代码的 scores 位于 NPU，因此 `torch.topk` 也在 NPU 上执行，Host 负责发起调用。本实践为何复用 PyTorch 的 TopK，而不自行编写 SIMT TopK？（提示：并行选择、排序及同分处理的复杂度，不能用一次 atomic_max 代替 TopK。）


In [ ]:
# 在此完成实践二
pass

## 4. 编程实践三（困难）：Scatter + 去重

**任务**：把 [512, 64] 的数据按索引散射到 [1024, 64] 的目标区，索引可能重复；重复时**保留分数最大者，同分取原始来源行号较小者**。输入 score 限定为有限 FP32，未命中的输出行约定为零。

**提示**：采用三个独立 Kernel 阶段：① atomic_max 求目标行最大分数；② 对最终最大分数的并列来源用 atomic_min 选最小来源行号；③ 仅唯一胜者写入。三个阶段在同一 Stream 顺序启动（或显式建立事件依赖），不能用块内 syncthreads 冒充全 Grid 屏障。score 与来源行号分开无保护写入会产生竞争。


In [ ]:
# 在此完成实践三
pass

In [ ]:
!cat ./answer/07.05_practice_answer.txt

### 完整参考实现与验收

包含两种输出尺寸的 Gather、带非法桶号保护的频数统计，以及三阶段确定性 Scatter。校验覆盖重复目标、同分、负分、未命中行和边界目标，并逐元素检查 winner；PyTorch TopK 的同分顺序由同一份选中索引传给 Golden，不额外声称稳定。

[参考源码](./src/irregular_examples.py) 可在 950 上从章节目录执行 `python src/irregular_examples.py`。下列单元仅展示源码；设备编译、精度与性能均以实际执行为准。


In [ ]:
!cat ./src/irregular_examples.py


## 5. 本节小结

Gather/Scatter/竞争仲裁覆盖了不规则数据的完整谱系——第 11 章的 Gated-KV SDPA（按门控分数选择 KV 块）正是"top-k + Gather + 竞争仲裁"的矩阵级组合。下一章回到全算子视角：调试与性能优化。